<center><a href="https://www.nvidia.com/en-us/training/"><img src="../images/nvidia_header.png" width="400"/></a></center>

# 0. Course Onboarding

Welcome to **Build and Evaluate a Visual AI Agent**. This notebook gets your lab and your tools ready for Part 1. Part 1 runs on one NVIDIA Video Search and Summarization (VSS) 3.2.1 deployment. RT-CV detects and tracks objects with TrafficCamNet RT-DETR, RT-VLM describes video with Cosmos 3 Super Reasoner, and RT-Embed makes video searchable with Cosmos-Embed1.

In this notebook you will:

- run a cell and open a terminal in JupyterLab
- read the plan for the day and see where each part runs
- check that the VSS services are ready and the search videos are indexed
- sign in to Codex, the coding agent you use at the end of Part 1
- call the services through `http://nginx/`, not `localhost`

**Time:** 15 minutes

<!-- generated by task1/scripts/course_path.py from part1/course_path.json; edit the JSON, not this cell -->
> **Introduction** &middot; about 15 min &nbsp;|&nbsp; [Home](../00_Home.ipynb)
>
> Next: [1. Inspect VSS Deployment](01-inspect-vss-deployment.ipynb)


## 0.1 Run a cell in JupyterLab

JupyterLab has a menu bar at the top, a file browser on the left, and a work area where notebooks and terminals open as tabs. A notebook is a list of cells. Text cells like this one hold the instructions, and code cells run Python.

To run a cell, click it and press **Shift+Enter**. If a text cell switches to edit mode, press **Shift+Enter** to show it again. To open a terminal, choose **File → New → Terminal**. You need one in section 0.5.

Run the cell below now. It prints the time and confirms that the kernel works.

In [ ]:
# Select this cell and press Shift+Enter.
import datetime

print("Cell ran at", datetime.datetime.now(datetime.timezone.utc).strftime("%H:%M:%S UTC"))
print("The kernel works. Next: the plan for the day, then the readiness check.")

## 0.2 Read the plan for the day

The day has an introduction, two parts, and lunch in between. Part 1 runs in this lab. Part 2 runs on a second launch, a new lab that you start before lunch. The course page runs only one lab at a time, so you stop this lab first.

<img src="../images/course/day-map.gif" width="960" alt="Animated map of the day. After the VSS presentation, the first launch runs 0 Onboarding, then Part 1: 1 Inspect VSS Deployment, 2 RT-CV, 3 RT-VLM, 4 RT-Embed, the Part 1 assessment and VSS coding agent skills. At lunch you stop Part 1 and start Part 2. The second launch runs Part 2: 2.1 zero shot, 2.2 synthetic data, 2.3 VLM fine-tuning, 2.4 fine-tuned VSS and the Part 2 assessment. The minutes sit under each step, and chips mark the coding-agent steps and the assessments.">

| Part | Step | How | Minutes |
|---|---|---|---|
| Introduction | VSS presentation | slides | 15 |
| | 0. Course Onboarding | this notebook | 15 |
| Part 1: VSS Building Blocks, in this lab | [1. Inspect VSS Deployment](01-inspect-vss-deployment.ipynb) | notebook | 15 |
| | [2. RT-CV: Detection & Tracking](02-rt-cv-detection-and-tracking.ipynb) | notebook | 40 |
| | [3. RT-VLM: Captioning & Prompting](03-rt-vlm-captioning-and-prompting.ipynb) | notebook | 35 |
| | [4. RT-Embed: Video Search](04-rt-embed-video-search.ipynb) | notebook | 30 |
| | [Part 1 Assessment](assessment.ipynb) | notebook, on your own | 25 |
| | VSS Coding Agent Skills, a page in Course Docs | coding agent | 40 |
| Lunch | Keep a copy of your work, click STOP TASK for Part 1, then START for Part 2 | | 45 |
| Part 2: Applying VSS to Traffic Anomalies, on the second launch | 2.1 VSS for Traffic Anomalies: Zero Shot | coding agent | 35 |
| | 2.2 Synthetic Data Generation with PAIDF | coding agent | 30 |
| | 2.3 VLM Fine-Tuning | notebook | 45 |
| | 2.4 VSS Fine-Tuned: Deploy and Evaluate | coding agent | 40 |
| | Part 2 Assessment | coding agent, on your own | 30 |

Part 1 takes 195 minutes and Part 2 takes 210, with a 10-minute break in each and short slides in Part 2. The instructor gives the real times.

Files in this lab do not carry over to the Part 2 lab, where you deploy VSS yourself with the coding agent. Before lunch, run the cell under **Keep a copy of your work (Save HTMLs)** on the [Home page](../00_Home.ipynb) and download the zip it links. Then, on the course page, click **STOP TASK** for Part 1 and **START** for Part 2. Part 2 cannot start while this lab runs.

Part 2 also needs your own NGC Personal API Key. If you do not have one yet, create it at [ngc.nvidia.com](https://ngc.nvidia.com) before Part 2 starts, with
**NGC Catalog** selected under **Services Included**. NGC shows a new key only once, so save it somewhere safe.

## 0.3 Wait until the lab is ready

When your lab launched, a deployer service started the VSS services. On a new lab they can still be starting. The cell below checks the seven endpoints and the eleven containers the course uses, through `http://nginx` and the Docker daemon behind the lab. It checks every 15 seconds for up to 20 minutes. It prints a service's line when its status changes, and one `NOT READY` line on each pass while it waits.

What to look for:

- On a lab that has been up for a while: `READY TO CONTINUE` after one pass.
- On a lab that just launched: `WAIT / CHECK` lines that turn into `PASS` one by one. While you wait, go to section 0.5 and sign in to Codex in a terminal. Come back when the cell stops.
- A `RuntimeError` means the deployer reported `FAILED`. Call the instructor. Waiting will not help.
- A `TimeoutError` means some services were still not ready after 20 minutes. If you launched your lab less than half an hour ago, run the cell once more. Otherwise, call the instructor before you go on.

In [ ]:
# Readiness check: repeats every 15 s for up to 20 minutes and raises if the lab is not ready.
# Prints each status line when it changes. Kibana is shown but not required.
import os
import subprocess
import time

import requests

from course_setup import status_tail

BASE = "http://nginx"
SEED_STATUS = "/opt/vss/_course-runtime/seed-status"  # deployer lines "<ISO time> RUNNING|OK|FAILED: ..."; the LAST line is the current state
POLL_S, CAP_S = 15, 20 * 60

SERVICES = {
    "RT-CV":         "/rtvi-cv/api/v1/ready",
    "RT-VLM":        "/rtvi-vlm/v1/health/ready",
    "RT-Embed":      "/rtvi-embed/v1/ready",
    "VIOS":          "/vios/vst/api/v1/sensor/list",
    "Elasticsearch": "/es/_cluster/health",
    "Kibana":        "/kibana/",
    "VSS agent":     "/agent/health",
}
OPTIONAL = {"Kibana"}
CONTAINERS = [
    "vss-agent", "vss-rtvi-cv", "vss-rtvi-vlm", "vss-rtvi-embed",
    "vss-vios-ingress", "vss-vios-sensor", "vss-vios-streamprocessing",
    "elasticsearch", "kafka", "logstash", "vss-behavior-analytics",
]


def probe_once():
    """One pass over the endpoints and containers. Returns ({name: status line}, [names not ready])."""
    lines, pending = {}, []
    for name, path in SERVICES.items():
        try:
            response = requests.get(BASE + path, timeout=10, allow_redirects=False)
            ok = response.status_code == 200 or (name == "Kibana" and response.status_code in (301, 302))
            detail = f"HTTP {response.status_code}"
            if name == "Elasticsearch" and response.status_code == 200:
                health = response.json()
                ok = health.get("status") in ("green", "yellow") and not health.get("timed_out", False)
                detail += f", cluster {health.get('status')}"
        except (requests.RequestException, ValueError) as exc:
            ok, detail = False, type(exc).__name__
        label = "PASS" if ok else ("OPTIONAL" if name in OPTIONAL else "WAIT / CHECK")
        lines[name] = f"{label:12} {name:27} {detail}"
        if not ok and name not in OPTIONAL:
            pending.append(name)
    try:
        result = subprocess.run(
            ["docker", "inspect", "--format",
             "{{.Name}}|{{.State.Status}}|{{if .State.Health}}{{.State.Health.Status}}{{else}}no-healthcheck{{end}}",
             *CONTAINERS], capture_output=True, text=True, timeout=30,
        )
        states = {parts[0].lstrip("/"): parts[1:]
                  for line in result.stdout.splitlines() if len(parts := line.split("|")) == 3}
        for name in CONTAINERS:
            state, health = states.get(name, ("missing", "unknown"))
            ok = state == "running" and health in ("healthy", "no-healthcheck")
            lines[name] = f"{'PASS' if ok else 'WAIT / CHECK':12} {name:27} {state}, {health}"
            if not ok:
                pending.append(name)
        if result.returncode and not pending:
            pending.append("Docker inspection failed")
    except (OSError, subprocess.TimeoutExpired) as exc:
        pending.append(f"Docker unavailable ({type(exc).__name__})")
    return lines, pending


def seed_failed():
    """True when the deployer's current (last) status line reads FAILED. No file yet means the deployer has not started."""
    if not os.path.isfile(SEED_STATUS):
        return False
    with open(SEED_STATUS) as fh:
        lines = [line.strip() for line in fh.read().splitlines() if line.strip()]
    if not lines:
        return False
    _stamp, _, state = lines[-1].partition(" ")   # one line per step, appended; earlier lines are history
    return state.startswith("FAILED")


started = time.monotonic()
shown, last_tail, lab_ready = {}, None, False
while True:
    elapsed = int(time.monotonic() - started)
    lines, pending = probe_once()
    for name, line in lines.items():
        if shown.get(name) != line:
            print(f"[{elapsed:4d}s] {line}")
            shown[name] = line
    if not pending:
        lab_ready = True
        break
    tail = status_tail()
    if tail != last_tail:
        print(f"[{elapsed:4d}s] deployer status:\n    " + tail.replace("\n", "\n    "))
        last_tail = tail
    if seed_failed():
        raise RuntimeError("NOT READY: the deployer reported FAILED (status above). Waiting will not help. "
                           "Call the instructor now and show them this output.")
    print(f"[{elapsed:4d}s] NOT READY; waiting on: {', '.join(pending)}")
    if elapsed + POLL_S >= CAP_S:
        raise TimeoutError(f"NOT READY after {elapsed} s (cap {CAP_S} s); still waiting on: {', '.join(pending)}. "
                           "If you launched your lab less than half an hour ago, run this cell once more. "
                           "Otherwise, call the instructor before you go on.")
    time.sleep(POLL_S)

print(f"\nREADY TO CONTINUE after {int(time.monotonic() - started)} s: the required endpoints and containers passed.")
print("Next: section 0.4 checks that the search videos are indexed.")

`READY TO CONTINUE` means every required endpoint answers and all eleven containers are running, with their health checks passing where they have one. Elasticsearch may report `yellow` because this lab runs a single node. That passes, but `red` does not. Kibana is checked too, but the cell does not wait for it: if it does not answer, its line says `OPTIONAL`. An HTTP 502, a `ConnectionError` or a container marked `starting` means the stack is still coming up.

While the cell waits, it also prints the deployer's status. The first line is the latest entry in `/opt/vss/_course-runtime/seed-status`, where the deployer writes one line per launch step. The lines below it are the end of the deployer's log, `seed.log`. If that latest status line reads `FAILED`, the cell stops right away with a `RuntimeError`.

`READY TO CONTINUE` does not mean the search videos are indexed yet. The next section checks that.

## 0.4 Check that the search videos are indexed

When your lab launched, a one-time service called `course-prep` started preparing the videos you search in notebook 4. It adds four three-minute traffic videos to VIOS (video storage), records their VIOS ids for notebook 4, and waits until RT-Embed (video search) has indexed all of them. It writes its progress, video by video, to `/dli/task/data/course_prep_status.json`.

Run the cell below. It prints that file and stops with an error unless `state` is `OK`:

- `RUNNING`: course-prep is still working. Carry on with this notebook and notebooks 1 to 3, and run this cell again before you start notebook 4.
- `FAILED`: call the instructor.
- File missing: course-prep has not started yet. Run the cell again in a minute.

Notebook 4 and the Part 1 assessment search these four videos. In VSS Coding Agent Skills, the coding agent adds its own copies of the same four files. The state must read `OK` before you start notebook 4.

In [ ]:
# course-prep status: prints it, and raises unless it reads OK.
import json
import os

PREP_STATUS = "/dli/task/data/course_prep_status.json"  # written by the course-prep service

if not os.path.isfile(PREP_STATUS):
    raise RuntimeError(f"{PREP_STATUS} does not exist yet: the course-prep service has not started. "
                       "Run this cell again in a minute. If it is still missing ten minutes after the readiness "
                       "check said READY, call the instructor.")
with open(PREP_STATUS) as fh:
    prep = json.load(fh)

print(f"state: {prep['state']}   updated: {prep.get('updated')}")
if prep.get("detail"):
    print(f"detail: {prep['detail']}")
for name, asset in prep["assets"].items():
    print(f"  {asset['state']:8} {name:28} sensor {asset.get('sensor_id') or '-'}  {asset.get('error') or ''}")

if prep["state"] == "RUNNING":
    raise RuntimeError("course-prep is still RUNNING. Carry on with this notebook and notebooks 1 to 3, "
                       "and run this cell again before you start notebook 4.")
if prep["state"] != "OK":
    raise RuntimeError(f"course-prep reports {prep['state']}: call the instructor and show them the lines above. "
                       "Waiting will not help.")
print("\nPREPARATION OK: the four search videos are in VIOS and indexed for search.")

## 0.5 Sign in to Codex now

Codex is a coding agent: a tool you talk to in a terminal while you build and test. At the end of Part 1, you ask it to do the work of notebooks 2 to 4 for you, using the VSS skills, and to watch a live stream for a collision. It is not part of the deployed VSS pipeline, which runs without it.

Sign in now, so that a sign-in problem shows up early and not at the end of Part 1. The account to use is in the course prerequisites you were given. Part 1 needs no API key.

1. Open a terminal: **File → New → Terminal**.
2. Start the device login:

   ```bash
   codex login --device-auth
   ```

3. The terminal prints a URL and a one-time code. Open the URL in a new tab of your laptop's browser. Sign in and enter the code. Never share the code. Device codes are a common target for phishing.

   If the page does not accept the code, turn on device-code sign-in for your account. In another browser tab, sign in at auth.openai.com, open **Settings → Security** and turn on **Enable device code authorization for Codex**. Then enter the code again.

4. Check the login:

   ```bash
   codex login status
   ```

   It should say that you are logged in. Leave the terminal open for section 0.6. The login is saved, so a new terminal is signed in too.
5. Raise your hand if you see an error instead of a URL, the page still refuses the code after you turned that setting on, or `codex login status` does not say that you are logged in. Do not retry in a loop, and do not paste a key anywhere.

You start Codex itself at the end of Part 1, in VSS Coding Agent Skills. That step is a page in Course Docs, not a notebook: open the JupyterLab Launcher (**File → New Launcher**) and click **Course Docs**. Part 2 runs on a new launch, and you sign in again there.

## 0.6 Call services through `nginx`, not `localhost`

<img src="../images/course/where-things-run.png" width="960" alt="Your laptop's browser reaches nginx, the course proxy, through the lab's public hostname, and the notebook kernel, terminal and Codex in the lab container reach it as http://nginx/..., while localhost only points back at the lab container, where no service runs. nginx routes each path to vss-agent, VIOS, RT-CV, RT-Embed, RT-VLM or Elasticsearch, and docker reaches the deployment's Docker daemon through DOCKER_HOST, with a note never to down or prune it.">

Your notebook and your terminal run in the lab container. The VSS services run next to it, behind `nginx`, the course proxy. From your notebook and your terminal, every service address starts with `http://nginx/`, for example `http://nginx/agent/health` or `http://nginx/es/_cat/indices`. `localhost` points back at the lab container, so `Connection refused` on `localhost` tells you nothing about VSS. Your laptop's browser reaches the same services through the lab's public hostname.

The `docker` command in your terminal talks to the deployment's Docker daemon through `DOCKER_HOST`. Use it to look at containers, for example with `docker ps`. Never use it to stop or remove them. Never print a container's environment either, for example with a plain `docker inspect`, because it holds the course key.

In your terminal, run these three checks. They ask RT-CV (detection and tracking), RT-VLM (captioning) and RT-Embed whether they are ready.

```bash
curl -s -w '  HTTP %{http_code}\n' http://nginx/rtvi-cv/api/v1/ready
curl -s -w '  HTTP %{http_code}\n' http://nginx/rtvi-vlm/v1/health/ready
curl -s -w '  HTTP %{http_code}\n' http://nginx/rtvi-embed/v1/ready
```

Each check ends with `HTTP 200` when that service is ready. That is the answer the readiness check in section 0.3 waited for.

## Check your understanding

1. The readiness check says `READY TO CONTINUE`. Are the four search videos indexed?
2. In the terminal, `curl http://localhost/agent/health` fails. Is the VSS agent service down?
3. The deployer status reads `FAILED: ...`. What do you do?

<details>
<summary>Answers</summary>

1. Not necessarily. `READY TO CONTINUE` means the services answer and the containers are healthy. Indexing is the job of course-prep, and section 0.4 checks it.
2. No. `localhost` is your lab container, and no service runs there. The VSS agent service answers at `http://nginx/agent/health`.
3. Call the instructor. The deployer writes `FAILED` when a launch step fails and it cannot recover, so waiting will not change it.

</details>

## Next

Open [1. Inspect VSS Deployment](01-inspect-vss-deployment.ipynb). You look at the services and models in the deployment and at the path video data takes through them. Keep your terminal open.

If the readiness check did not say `READY TO CONTINUE`, tell the instructor before you go on.

<!-- generated by task1/scripts/course_path.py from part1/course_path.json; edit the JSON, not this cell -->
> **You are done with this step when:** the readiness check in 0.3 prints `READY TO CONTINUE`.
>
> **Next: [1. Inspect VSS Deployment](01-inspect-vss-deployment.ipynb)** &middot; Part 1 &middot; about 15 min  
> See which services and models run here, check that they answer, and read the detector's labels.
>
> [Back to Home](../00_Home.ipynb)


<center><a href="https://www.nvidia.com/en-us/training/"><img src="../images/nvidia_header.png" width="400"/></a></center>